In [7]:
from ASKpipeline import build_verification_graph
import pandas as pd

In [8]:
from typing import Dict, List
import pandas as pd

def run_verification_on_datasets(
    datasets: Dict[str, pd.DataFrame],
    question_col: str = "question",
    answer_col: str = "answer",
    build_graph_fn=None,
    show_progress: bool = True,
) -> Dict[str, List[str]]:
    """
    Runs the verification graph on each row of each dataframe.
    Returns: {dataset_name: [verdicts...]} in row order.
    """
    if build_graph_fn is None:
        from ASKpipeline import build_verification_graph
        build_graph_fn = build_verification_graph

    graph = build_graph_fn()
    results: Dict[str, List[str]] = {}

    for name, df in datasets.items():
        verdicts: List[str] = []
        iterator = df.itertuples(index=False)
        if show_progress:
            try:
                from tqdm import tqdm
                iterator = tqdm(list(iterator), desc=f"Verifying {name}")
            except Exception:
                pass

        for row in iterator:
            row_dict = row._asdict()
            state = {
                "question": row_dict.get(question_col, ""),
                "answer": row_dict.get(answer_col, ""),
            }
            try:
                out = graph.invoke(state)
                verdicts.append(out.get("verdict", "Not Found"))
            except Exception:
                verdicts.append("ERROR")

        results[name] = verdicts

    return results


In [9]:
names = ['mintaka', 'qald', 'hotpot']

variants = ['small','base','large']
norm_files = [f'flan-t5-{variant}' for variant in variants]
vanilla_files = [f'vanilla-flan-t5-{variant}' for variant in variants]

files = norm_files + vanilla_files

file_name = files[0]


dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}

if file_name.startswith('vanilla'):
    dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}




In [10]:
from ASKpipeline import build_verification_graph

state = {"queries": 'ASK WHERE {{ wd:Q5351150 wdt:P17 wd:Q49 . }UNION{ wd:Q49 wdt:P17 wd:Q5351150 . }}'}
graph = build_verification_graph(state)

row = dataframes["mintaka"].iloc[0]
state = {"question": row["SAE Question"], "answer": row["Answer"]}
out = graph.invoke(state)



parsed_entities: {0: ['El Diente Peak', 'North America']}
parsed_relations: {0: ['tallest mountain', 'location', 'part of mountain range', 'country', 'elevation']}
entity_qids: {0: ['Q5351150', 'Q49']}
relation_pids: {0: ['P625', 'P17', 'P2044']}
queries: ['ASK WHERE {{ wd:Q5351150 wdt:P625 wd:Q49 . }UNION{ wd:Q49 wdt:P625 wd:Q5351150 . }}']
results: [False]


In [11]:
type(out['results'][0])

bool

In [ ]:
from collections import Counter

all_results = {}

for file_name in files:
    if file_name.startswith('vanilla'):
        dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}
    else: 
        dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}
    

    file_results = {}
    for name, data in dataframes.items():
        print(f'Starting verification for {name}')
        row_results = []
        all_votes = []

        for _, row in data.iterrows():
            state = {'question': row["SAE Question"], "answer": row["Answer"]}
            out = graph.invoke(state)
            results = out.get("results") or []
            row_results.append([str(r) for r in results])

            all_votes.extend([str(r) for r in results])

        majority = Counter(all_votes).most_common(1)[0][0] if all_votes else "None"
        print(majority)

        fin_res = pd.DataFrame({
            "Results": row_results,
            "Majority": [majority] * len(row_results),
        })

        file_results[name] = fin_res
    path = f'./final_results/{file_name}/{name}.csv'
    file_results.to_csv(path, Index = False) #intermittent saving

    all_results[file_name] = file_results


Starting verification for mintaka
parsed_entities: {0: ['El Diente Peak', 'North America']}
parsed_relations: {0: ['tallest mountain', 'location', 'part of mountain range', 'elevation', 'country', 'continent']}
entity_qids: {0: ['Q5351150', 'Q49']}
relation_pids: {0: ['P625', 'P2044', 'P17', 'P30']}
queries: ['ASK WHERE {{ wd:Q5351150 wdt:P625 wd:Q49 . }UNION{ wd:Q49 wdt:P625 wd:Q5351150 . }}']
results: [False]
parsed_entities: {0: ['Titanic', 'actor'], 1: ['Leonardo DiCaprio']}
parsed_relations: {0: [], 1: ['born in', 'starred in']}
entity_qids: {0: ['Q25173', 'Q33999'], 1: ['Q38111']}
relation_pids: {0: [], 1: ['P19']}
queries: [None, None]
results: [None, None]
parsed_entities: {0: ['actor', 'Vanilla Sky'], 1: ['Katie Holmes'], 2: []}
parsed_relations: {0: ['starring', 'film'], 1: ['P26'], 2: []}
entity_qids: {0: ['Q33999', 'Q42415916'], 1: ['Q174346'], 2: []}
relation_pids: {0: ['P161', 'P480'], 1: ['P26'], 2: []}
queries: ['ASK WHERE {{ wd:Q33999 wdt:P161 wd:Q42415916 . }UNION{ wd

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-93JpGdxiONDBkDORMHEm5Xfn on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}